# Exp 2 Reproducibility Check

This notebook reruns `code/02_experiment2_evaluation.R` in an isolated scratch workspace
and compares the rerun numeric outputs against `expected/exp2_evaluation.rds` and the
curated modeling data in `data/model_df.rds`.

In [ ]:
suppressPackageStartupMessages({
  library(survey)
  library(tidyverse)
  library(pROC)
})

helper_candidates <- c("repro_utils.R", file.path("notebooks", "repro", "repro_utils.R"))
helper_path <- helper_candidates[file.exists(helper_candidates)][1]
if (is.na(helper_path)) {
  stop("repro_utils.R not found.")
}
source(helper_path)

baseline_result_path <- path_in_repo("expected", "exp2_evaluation.rds")
baseline_model_df_path <- path_in_repo("data", "model_df.rds")
scratch_dir <- new_scratch("exp2_repro")

cat("Scratch workspace:", scratch_dir, "\n")
cat("Baseline result:", baseline_result_path, "\n")

In [ ]:
run_script_in_scratch(
  "code/02_experiment2_evaluation.R",
  scratch_dir,
  c("data/nhanes_processed.rds", "data/survey_design.rds")
)

baseline_exp2 <- readRDS(baseline_result_path)
rerun_exp2 <- readRDS(file.path(scratch_dir, "results", "exp2_evaluation.rds"))
baseline_model_df <- readRDS(baseline_model_df_path)
rerun_model_df <- readRDS(file.path(scratch_dir, "results", "model_df.rds"))

In [ ]:
exp2_checks <- list(
  compare_df_tol("model_df", rerun_model_df, baseline_model_df, tol = 1e-12),
  compare_numeric_vec("auc_unwt", rerun_exp2$auc_unwt, baseline_exp2$auc_unwt, tol = 1e-10),
  compare_numeric_vec("auc_wt", rerun_exp2$auc_wt, baseline_exp2$auc_wt, tol = 1e-10),
  compare_numeric_vec("difference", rerun_exp2$difference, baseline_exp2$difference, tol = 1e-10),
  compare_numeric_vec("ci_unwt", rerun_exp2$ci_unwt, baseline_exp2$ci_unwt, tol = 1e-10),
  compare_numeric_vec("ci_wt", rerun_exp2$ci_wt, baseline_exp2$ci_wt, tol = 1e-10),
  compare_exact_scalar("n_boot", rerun_exp2$n_boot, baseline_exp2$n_boot),
  compare_numeric_vec("boot_auc", rerun_exp2$boot_auc, baseline_exp2$boot_auc, tol = 1e-10)
)

bind_rows(exp2_checks)

In [ ]:
exp2_summary <- summarize_results(exp2_checks)
exp2_summary$results